In [16]:
import wandb
from bibliotecas_externas.seq2seqvc.seq2seq_vc.losses import Seq2SeqLoss, DurationPredictorLoss, \
    StochasticDurationPredictorLoss

from main.arquitetura.Seq2SeqVC.models.model import seq2seq_AASVC

from main.dataloader.CVMPT.CVMPT_offline import CVMPT_offline

import torch



In [17]:
%load_ext autoreload
%autoreload 2

import wandb
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/mario/.netrc


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


True

In [18]:
yaml_model = r"/home/mario/Mestrado_VC/main/arquitetura/Seq2SeqVC/configs/AASVC_ENG/aas_vc.melmelmel.v1.yaml"
checkpoint_model_path = r"/home/mario/Mestrado_VC/main/arquitetura/Seq2SeqVC/configs/AASVC_JP/checkpoint-50000steps.pkl"

dataset_train_path = r"/home/mario/Mestrado_VC/dataset/cv-corpus-mozilla-pt/data/treinamento_gp"
dataset_val_path = r"/home/mario/Mestrado_VC/dataset/cv-corpus-mozilla-pt/data/teste_gp"

In [19]:
from main.arquitetura.Seq2SeqVC.datasets.collaters.nar_vc import NARVCCollater

device = "cuda" if torch.cuda.is_available() else "cpu"
model_seq2seq = seq2seq_AASVC(yaml_model,device)



l1_loss = torch.nn.L1Loss()
bce_loss = torch.nn.BCEWithLogitsLoss()


train_set = CVMPT_offline(path=dataset_train_path)
valid_set = CVMPT_offline(path=dataset_val_path)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=4,
    shuffle=True,
    collate_fn=NARVCCollater()
)

valid_loader = torch.utils.data.DataLoader(
    valid_set,
    batch_size=1,
    shuffle=False,
    collate_fn=NARVCCollater()
)


In [20]:
from bibliotecas_externas.seq2seqvc.seq2seq_vc.vocoder.griffin_lim import Spectrogram2Waveform

vocoder = Spectrogram2Waveform(
    n_fft=1024,
    n_shift=256,
    fs=22050,
    n_mels=80,
    griffin_lim_iters=32,
    take_norm_feat=False,
    #stats=trg_stats,  # stats do target
)


In [21]:
from main.arquitetura.Seq2SeqVC.trainers.TrainerAASVCMod import Trainer
from bibliotecas_externas.seq2seqvc.seq2seq_vc.losses import L1Loss, ForwardSumLoss
# bibliotecas_externas.seq2seqvc.seq2seq_vc.trainers import ARVCTrainer, AASVCTrainer

# contadores
steps = 0
epochs = 0

# dataloaders
data_loader = {
    "train": train_loader,
    "dev": valid_loader,
}

# sampler (sem DDP)
sampler = {}

# modelo
model = model_seq2seq.to(device)

# loss
criterion = {
    "L1Loss": L1Loss(),
    "ForwardSumLoss": ForwardSumLoss(),
    "StochasticDurationPredictorLoss": StochasticDurationPredictorLoss(),
}

optimizer = torch.optim.Adam(
    model_seq2seq.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

# scheduler (opcional)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=4000,
    gamma=1.0
)

inference_args = {
    "threshold": 0.5,
    "minlenratio": 6.0,
    "maxlenratio": 0.0,
}

# config
config = {
    "outdir": "./experiments/aas_vc_test_inference",
    "train_max_steps": 50000,
    "log_interval_steps": 10,
    "eval_interval_steps": 10,
    "save_interval_steps": 5000,
    "distributed": False,
    "rank": 0,
    "gradient_accumulate_steps": 1,
    "grad_norm": 0,
    "num_save_intermediate_results": 4,
    "inference": inference_args,
    "criterions": ["L1Loss", "ForwardSumLoss", "StochasticDurationPredictorLoss"],
    "lambda_align": 2.0,
    "dp_train_start_steps":0
    
}

# trainer
trainer = Trainer(
    steps=steps,
    epochs=epochs,
    data_loader=data_loader,
    sampler=sampler,
    model=model,
    vocoder=vocoder,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    config=config,
    is_test=False,
    device=device,
)
checkpoint_model_path = "/home/mario/Mestrado_VC/experiments/aas_vc_3_100k/checkpoint-250000steps.pkl"
trainer.load_checkpoint(checkpoint_model_path)

In [22]:

for i in valid_loader:
    mel = i["xs"][0]
    dp = i["dp_inputs"][0]
    break

inferencia = trainer.model.inference(torch.Tensor(mel).cuda(),dp_input=torch.Tensor(dp).cuda() )

from main.utils.visualizar import vizualizar_spectrogram

vizualizar_spectrogram(inferencia[0])


In [23]:
# project="Laringe Eletronica Seq2Seq - Mestrado VC"
# 
# config = {
#     "epocas": 100
# }
# trainer.run_wandb(project,config)

In [24]:
# import os
# import imageio
# import re
# pasta_raiz = r"/home/mario/Mestrado_VC/experiments/aas_vc_3_100k/predictions"
# 
# 
# 
# # ordenar as pastas (epocas)
# epocas = sorted([
#     p for p in os.listdir(pasta_raiz)
#     if os.path.isdir(os.path.join(pasta_raiz, p))
# ])
# 
# # pegar nomes das imagens da primeira epoca
# primeira_epoca = os.path.join(pasta_raiz, epocas[0])
# imagens = [f for f in os.listdir(primeira_epoca) if f.endswith("inference.png")]
# 
# for img_nome in imagens:
# 
#     frames = []
# 
#     for epoca in epocas:
# 
#         caminho = os.path.join(pasta_raiz, epoca, img_nome)
# 
#         if os.path.exists(caminho):
#             frames.append(imageio.imread(caminho))
# 
#     if frames:
#         imageio.mimsave(f"{img_nome.replace('.png','')}.gif", frames, duration=0.5)
# 
#         print(f"GIF criado: {img_nome}")